# 05 Failure Simulation

## Objective
To Simulate the removal of each station from the London Underground network and measure the resulting disruption.

## Inputs
- `data/processed/stations_clean.csv`
- `data/processed/connections_clean.csv`

## Outputs
- `data/processed/failure_results.csv`

## Disruption metrics
- Number of connected components
- Largest remaining component size
- Average shortest path
- Failure impact score

## Why this step matters
Centrality alone does not fully capture operational vulnerability. Simulated station removal provides a more realistic view of how failures propagate through the network.

In [11]:
import pandas as pd
import networkx as nx
from pathlib import Path

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

In [12]:
stations_clean = pd.read_csv(PROCESSED_DIR / "stations_clean.csv")
connections_clean = pd.read_csv(PROCESSED_DIR / "connections_clean.csv")

In [13]:
used_station_names = set(connections_clean["from_name"]).union(set(connections_clean["to_name"]))
stations_filtered = stations_clean[stations_clean["station_name"].isin(used_station_names)].copy()

G2 = nx.Graph()

for _, row in stations_filtered.iterrows():
    G2.add_node(
        row["station_name"],
        station_id=row["station_id"],
        lat=row["lat"],
        lon=row["lon"]
    )

for _, row in connections_clean.iterrows():
    G2.add_edge(
        row["from_name"],
        row["to_name"],
        line=row["line"]
    )

largest_component_nodes = max(nx.connected_components(G2), key=len)
G_main = G2.subgraph(largest_component_nodes).copy()

print("Nodes in main graph:", G_main.number_of_nodes())
print("Edges in main graph:", G_main.number_of_edges())
print("Connected components in main graph:", nx.number_connected_components(G_main))

Nodes in main graph: 272
Edges in main graph: 314
Connected components in main graph: 1


## Step 1: Define failure simulation functions

Two helper functions are used:
- one to remove a station from the graph
- one to measure the post-failure network structure

In [14]:
def simulate_station_failure(graph, station_name):
    G_failed = graph.copy()
    if station_name in G_failed:
        G_failed.remove_node(station_name)
    return G_failed


def get_network_stats(graph):
    if graph.number_of_nodes() == 0:
        return {
            "num_components": 0,
            "largest_component_size": 0,
            "avg_shortest_path": None
        }

    components = list(nx.connected_components(graph))
    num_components = len(components)
    largest_component = max(components, key=len)
    largest_subgraph = graph.subgraph(largest_component)

    if largest_subgraph.number_of_nodes() > 1:
        avg_shortest = nx.average_shortest_path_length(largest_subgraph)
    else:
        avg_shortest = None

    return {
        "num_components": num_components,
        "largest_component_size": len(largest_component),
        "avg_shortest_path": avg_shortest
    }

## Step 2: Measure the baseline network

This establishes the normal state of the network before simulating station failures.

In [15]:
baseline = get_network_stats(G_main)
baseline

{'num_components': 1,
 'largest_component_size': 272,
 'avg_shortest_path': 13.962448448013891}

## Step 3: Simulate station removal for every node

Each station is removed one at a time and the resulting network structure is recorded.

In [16]:
results = []

for station in G_main.nodes():
    G_failed = simulate_station_failure(G_main, station)
    failed_stats = get_network_stats(G_failed)

    results.append({
        "station_removed": station,
        "num_components": failed_stats["num_components"],
        "largest_component_size": failed_stats["largest_component_size"],
        "avg_shortest_path": failed_stats["avg_shortest_path"]
    })

failure_results = pd.DataFrame(results)
failure_results.head()

,station_removed,num_components,largest_component_size,avg_shortest_path
0,amersham,1,271,13.912396
1,chalfont and latimer,3,269,13.816484
2,chesham,1,271,13.912396
3,croxley,2,270,13.897535
4,chorleywood,2,268,13.778160


In [17]:
def minmax(series):
    denom = series.max() - series.min()
    if denom == 0:
        return pd.Series(0, index=series.index)
    return (series - series.min()) / denom


failure_results["components_norm"] = minmax(failure_results["num_components"])

failure_results["largest_component_loss"] = (
    failure_results["largest_component_size"].max() - failure_results["largest_component_size"]
)

failure_results["largest_component_loss_norm"] = minmax(failure_results["largest_component_loss"])
failure_results["avg_shortest_path_norm"] = minmax(failure_results["avg_shortest_path"])

## Step 4: Create a failure impact score

A composite score is built using:
- network fragmentation
- loss of the largest connected component
- change in route efficiency

In [18]:
failure_results["failure_impact_score"] = (
    0.4 * failure_results["components_norm"] +
    0.3 * failure_results["largest_component_loss_norm"] +
    0.3 * failure_results["avg_shortest_path_norm"]
)

failure_results = failure_results.sort_values("failure_impact_score", ascending=False)
failure_results.head(15)

,station_removed,num_components,largest_component_size,avg_shortest_path,components_norm,largest_component_loss,largest_component_loss_norm,avg_shortest_path_norm,failure_impact_score
47,camden town,3,251,13.667825,1.0,20,0.909091,0.093895,0.700896
57,earls court,3,262,14.673101,1.0,9,0.409091,0.354142,0.628970
68,finsbury park,3,259,13.870522,1.0,12,0.545455,0.146369,0.607547
164,stockwell,3,261,13.866578,1.0,10,0.454545,0.145349,0.579968
222,euston,2,249,13.731021,0.5,22,1.000000,0.110255,0.533077
5,moor park,3,264,13.646301,1.0,7,0.318182,0.088323,0.521951
69,finchley central,3,266,13.774237,1.0,5,0.227273,0.121443,0.504615
239,kings cross st. pancras,2,254,14.127852,0.5,17,0.772727,0.212987,0.495714
257,stratford,2,251,13.407044,0.5,20,0.909091,0.026384,0.480642
1,chalfont and latimer,3,269,13.816484,1.0,2,0.090909,0.132380,0.466987


In [19]:
failure_results["station_display"] = (
    failure_results["station_removed"]
    .astype(str)
    .str.replace(" underground", "", regex=False)
    .str.title()
)

## Step 5: Save station failure results

In [20]:
failure_results.to_csv(PROCESSED_DIR / "failure_results.csv", index=False)
print("Saved corrected failure_results.csv")

Saved corrected failure_results.csv
